In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
from torch.utils.data import DataLoader, Dataset, random_split
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import copy

import os
import kagglehub
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!mkdir -p ./results

In [5]:
# Download latest version
path = kagglehub.dataset_download("hamzamohiuddin/isbi-2012-challenge")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'isbi-2012-challenge' dataset.
Path to dataset files: /kaggle/input/isbi-2012-challenge


In [19]:
def get_transforms(img_size: int = 512, mode: str = "train"):
    """
    Args
    ----
    img_size : spatial size to pad/crop to (keep at 512 for ISBI)
    mode     : "train" (full augmentation) or "val"/"test" (no augmentation)
 
    Returns
    -------
    albumentations.Compose  — call as transform(image=np_img, mask=np_mask)
    Both inputs should be numpy arrays, dtype float32, values in [0, 1].
    """
    if mode == "train":
        return A.Compose([
            # ── Spatial transforms (applied to both image and mask) ──────────
 
            # Flip/rotate — free augmentation, always include
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
 
            # Elastic deformation — the key augmentation from the U-Net paper.
            # Simulates tissue deformation in EM images.
            # alpha controls deformation magnitude, sigma controls smoothness.
            A.ElasticTransform(
                alpha=34,       # magnitude of deformation (pixels)
                sigma=4,        # Gaussian smoothing (controls smoothness)
                p=0.5,
            ),
 
            # Random crop + resize — increases effective dataset size
            A.RandomResizedCrop(
                size=(img_size, img_size),
                scale=(0.8, 1.0),   # crop between 80-100% of image
                ratio=(0.9, 1.1),
                p=0.5,
            ),
 
            # GridDistortion — complements elastic transform
            A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
 
            # ── Intensity transforms (image ONLY — mask unaffected) ──────────
 
            # Brightness/contrast variation — simulates staining variability
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=0.5,
            ),
 
            # Gaussian noise — simulates EM detector noise
            A.GaussNoise(std_range=(0.1, 0.5), p=0.3),
 
            # Blur — simulates focus variation in microscopy
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MedianBlur(blur_limit=3, p=1.0),
            ], p=0.2),
 
            ToTensorV2(),   # → torch tensors, [C,H,W]
        ])
    else:
        # Val/test: only convert to tensor, no augmentation
        return A.Compose([ToTensorV2()])
 

In [18]:
class ISBIDataset(Dataset):
    """
    ISBI 2012 EM Segmentation Dataset.
    Identical directory layout to your original version; adds albumentations support.
 
    Args
    ----
    root_dir  : path from kagglehub (contains train/ and test/)
    split     : "train" or "test"
    transform : albumentations Compose object (from augmentation.get_transforms)
    """
 
    def __init__(self, root_dir: str, split: str = "train", transform=None):
        self.split     = split
        self.transform = transform
        self.img_dir   = os.path.join(root_dir, split, "imgs")
        self.label_dir = os.path.join(root_dir, split, "labels")
        self.images    = sorted(os.listdir(self.img_dir))
 
    def __len__(self):
        return len(self.images)
 
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        # Keep as float32 numpy [H,W] for albumentations
        image = np.array(Image.open(img_path).convert("L"), dtype=np.float32) / 255.0
 
        if self.split == "train":
            mask_path = os.path.join(self.label_dir, self.images[idx])
            mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32) / 255.0
            mask = (mask > 0.5).astype(np.float32)
 
            if self.transform:
                # albumentations expects HWC for images; we pass HW for grayscale
                augmented = self.transform(image=image, mask=mask)
                image = augmented["image"]  # → torch [1, H, W] float32 via ToTensorV2
                mask  = augmented["mask"]   # → torch [H, W] float32
            else:
                image = torch.from_numpy(image).unsqueeze(0)
                mask  = torch.from_numpy(mask)
 
            # Ensure float image, long mask for CrossEntropyLoss
            return image.float(), mask.long()
 
        else:
            if self.transform:
                augmented = self.transform(image=image, mask=np.zeros_like(image))
                image = augmented["image"]
            else:
                image = torch.from_numpy(image).unsqueeze(0)
            return image.float()

In [17]:
class DiceLoss(nn.Module):
    """
    Soft Dice loss for binary segmentation.
    Works on class-1 probability (membrane class).
    Better than pure CrossEntropy when foreground pixels are sparse.
    """
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: [B, 2, H, W]  targets: [B, H, W] long
        probs  = torch.softmax(logits, dim=1)[:, 1]   # probability of membrane
        targets_f = targets.float()
        intersection = (probs * targets_f).sum(dim=(1, 2))
        union        = probs.sum(dim=(1, 2)) + targets_f.sum(dim=(1, 2))
        dice         = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()
 
 
class CombinedLoss(nn.Module):
    """CrossEntropy + Dice, weighted by alpha."""
    def __init__(self, alpha: float = 0.5):
        super().__init__()
        self.ce    = nn.CrossEntropyLoss()
        self.dice  = DiceLoss()
        self.alpha = alpha
 
    def forward(self, logits, targets):
        return self.alpha * self.ce(logits, targets) + \
               (1 - self.alpha) * self.dice(logits, targets)
 
 
@torch.no_grad()
def dice_score(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds  = logits.argmax(dim=1).float()
    target = targets.float()
    inter  = (preds * target).sum()
    union  = preds.sum() + target.sum()
    return (2 * inter / (union + 1e-6)).item()

In [16]:
class DoubleConv(nn.Module):
    """
    Two consecutive (Conv → BatchNorm → ReLU) operations.
    This is the basic repeated unit in both encoder and decoder.

    Weight shapes (for SVD later):
        conv1.weight : (out_ch, in_ch, 3, 3)
        conv2.weight : (out_ch, out_ch, 3, 3)
    """

    def __init__(self, in_channels: int, out_channels: int, mid_channels: int = None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels

        self.block = nn.Sequential(OrderedDict([
            ("conv1", nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False)),
            ("bn1",   nn.BatchNorm2d(mid_channels)),
            ("relu1", nn.ReLU(inplace=True)),
            ("conv2", nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False)),
            ("bn2",   nn.BatchNorm2d(out_channels)),
            ("relu2", nn.ReLU(inplace=True)),
        ]))

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    """Encoder step: MaxPool → DoubleConv (halves spatial dims)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.pool(x))


class Up(nn.Module):
    """
    Decoder step: Upsample → concat skip → DoubleConv (doubles spatial dims).
    
    Skip connection arrives here as the second argument to forward().
    """

    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            # Bilinear upsample then halve channels with 1×1 conv
            self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            # Learned transposed convolution (also interesting to SVD)
            self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        # Pad if input is not perfectly divisible (odd spatial dims)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        x = F.pad(x, [dx // 2, dx - dx // 2, dy // 2, dy - dy // 2])

        # ── Skip connection concatenation ──
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    """
    Configurable U-Net.

    Parameters
    ----------
    in_channels   : input image channels (1 = grayscale, 3 = RGB)
    out_channels  : number of segmentation classes
    base_channels : feature channels at the first encoder level (doubles each level)
    depth         : number of down/up-sampling stages (encoder levels excluding bottleneck)
    bilinear      : True  → bilinear upsampling (fewer params)
                    False → transposed conv (learnable upsampling, more to SVD)

    Architecture (depth=4, base=64)
    --------------------------------
    Input → [64] → [128] → [256] → [512] → [1024] (bottleneck)
                                         ↕  skip[3]
                              [512] ← Up(1024+512)
                         ↕  skip[2]
                   [256] ← Up(512+256)
              ↕  skip[1]
        [128] ← Up(256+128)
    ↕  skip[0]
    [64]  ← Up(128+64)
    → Output conv → logits
    """

    def __init__(
        self,
        in_channels:   int  = 1,
        out_channels:  int  = 2,
        base_channels: int  = 64,
        depth:         int  = 4,
        bilinear:      bool = True,
    ):
        super().__init__()
        self.depth    = depth
        self.bilinear = bilinear

        # ── Encoder ──────────────────────────────────────────────────────────
        # enc[0] : initial conv (no pooling)
        # enc[1..depth-1] : Down blocks
        self.encoder = nn.ModuleList()
        self.encoder.append(DoubleConv(in_channels, base_channels))
        ch = base_channels
        for _ in range(depth - 1):
            self.encoder.append(Down(ch, ch * 2))
            ch *= 2

        # ── Bottleneck ───────────────────────────────────────────────────────
        factor = 2 if bilinear else 1
        self.bottleneck = Down(ch, ch * 2 // factor)
        ch = ch * 2 // factor

        # ── Decoder ──────────────────────────────────────────────────────────
        # Symmetric to encoder; skip from enc[depth-1] down to enc[0]
        self.decoder = nn.ModuleList()
        for i in range(depth - 1, 0, -1):
            skip_ch = base_channels * (2 ** i)
            self.decoder.append(Up(ch + skip_ch, skip_ch // factor, bilinear))
            ch = skip_ch // factor

        self.decoder.append(Up(ch + base_channels, base_channels, bilinear))

        # ── Output ───────────────────────────────────────────────────────────
        self.out_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        # Encode — store intermediate feature maps for skip connections
        skips = []
        for enc_block in self.encoder:
            x = enc_block(x)
            skips.append(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decode — consume skips in reverse order
        for dec_block, skip in zip(self.decoder, reversed(skips)):
            x = dec_block(x, skip)

        return self.out_conv(x)

    # ── Weight extraction helpers (for SVD) ──────────────────────────────────
 
    def get_weight_dict(self) -> dict:
        """
        Returns a flat dict of all conv weight tensors, keyed by human-readable names.
        Each value is a 2D numpy array (out_features × in_features) obtained by
        reshaping the 4D conv kernel (out, in, kH, kW) → (out, in*kH*kW).
 
        This 'unrolled' view is the natural matrix for computing singular values.
 
        Keys follow the pattern:
            "encoder_L{i}_conv{j}"   — encoder level i, conv layer j
            "bottleneck_conv{j}"
            "decoder_L{i}_conv{j}"   — decoder level i (from top of decoder)
        """
        weights = {}
 
        def extract_double_conv(prefix, double_conv: DoubleConv):
            for name in ["conv1", "conv2"]:
                w = getattr(double_conv.block, name).weight.detach().cpu()
                # Reshape (out, in, kH, kW) → (out, in*kH*kW)
                weights[f"{prefix}_{name}"] = w.reshape(w.shape[0], -1).numpy()
 
        # Encoder
        for i, block in enumerate(self.encoder):
            conv = block if isinstance(block, DoubleConv) else block.conv
            extract_double_conv(f"encoder_L{i}", conv)
 
        # Bottleneck
        extract_double_conv("bottleneck", self.bottleneck.conv)
 
        # Decoder
        for i, block in enumerate(self.decoder):
            extract_double_conv(f"decoder_L{i}", block.conv)
 
        return weights
 
    def save_weights_for_analysis(self, path: str):
        """
        Save structured weights to a .pt file.
        Loads back with:
            data = torch.load('weights.pt')
            data['weight_dict']   # flat dict of 2D numpy arrays
            data['model_state']   # full state dict to restore model
            data['config']        # hyperparameters
        """
        torch.save({
            "model_state": self.state_dict(),
            "weight_dict": self.get_weight_dict(),
            "config": {
                "depth":         self.depth,
                "bilinear":      self.bilinear,
            }
        }, path)
        print(f"Saved weights → {path}")

In [15]:
def snapshot_spectra(model: UNet) -> dict:
    """Compute and return SVD stats for every conv layer."""
    wdict   = model.get_weight_dict()
    results = {}
    for name, W in wdict.items():
        _, S, _ = np.linalg.svd(W, full_matrices=False)
        energy  = np.cumsum(S**2) / np.sum(S**2)
        results[name] = {
            "S":              S,
            "effective_rank": int(np.searchsorted(energy, 0.95)) + 1,
            "stable_rank":    float(np.sum(S**2) / S[0]**2) if S[0] > 0 else 0.0,
        }
    return results
 
 
def spectral_similarity_score(S_enc: np.ndarray, S_dec: np.ndarray) -> float:
    """
    Quantify how 'adjoint-like' a symmetric encoder/decoder pair is.
 
    Score = 1 - mean |σ_enc_i/σ_dec_i - 1|  (clamped to [0,1])
 
    Score of 1.0 → perfectly matched spectra (ideal adjoint).
    Score of 0.0 → completely unrelated spectra.
    """
    k = min(len(S_enc), len(S_dec))
    ratio = S_enc[:k] / (S_dec[:k] + 1e-10)
    score = 1.0 - float(np.mean(np.abs(ratio - 1.0)))
    return max(0.0, score)
 
 
def get_symmetric_pairs(spectra: dict) -> list[tuple[str, str]]:
    """
    Match encoder level i with decoder level (depth - 1 - i).
    Returns list of (enc_key, dec_key) sorted from shallowest to deepest.
    """
    enc_keys = sorted([k for k in spectra if k.startswith("encoder_")])
    dec_keys = sorted([k for k in spectra if k.startswith("decoder_")], reverse=True)
    return list(zip(enc_keys, dec_keys))

def plot_all(before: dict, after: dict, losses: list, out_dir: str = "."):
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from pathlib import Path
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
 
    # ── Training curve ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(losses) + 1)
    axes[0].plot(epochs, [l["train_loss"] for l in losses], label="train")
    axes[0].plot(epochs, [l["val_loss"]   for l in losses], label="val",
                 linestyle="--")
    axes[0].set(title="Loss", xlabel="Epoch", ylabel="Loss")
    axes[0].legend(); axes[0].grid(alpha=0.3)
 
    axes[1].plot(epochs, [l["val_dice"] for l in losses], color="green")
    axes[1].set(title="Validation Dice Score", xlabel="Epoch", ylabel="Dice")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out / "training_curve.png", dpi=150, bbox_inches="tight")
    plt.close()
 
    # ── Effective rank profile ─────────────────────────────────────────────────
    names    = list(before.keys())
    x        = np.arange(len(names))
    pre_eff  = [before[n]["effective_rank"] for n in names]
    post_eff = [after[n]["effective_rank"]  for n in names]
    delta    = [a - b for b, a in zip(pre_eff, post_eff)]
 
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(max(12, len(names)), 8), sharex=True)
    fig.suptitle("Effective Rank (95% energy): Before vs After Training")
    w = 0.4
    ax1.bar(x - w/2, pre_eff,  w, color="steelblue", alpha=0.8, label="init")
    ax1.bar(x + w/2, post_eff, w, color="tomato",    alpha=0.8, label="trained")
    ax1.set_ylabel("Effective rank"); ax1.legend(); ax1.grid(axis="y", alpha=0.3)
    colors = ["green" if d >= 0 else "red" for d in delta]
    ax2.bar(x, delta, color=colors, alpha=0.8)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_ylabel("Δ (trained − init)")
    ax2.set_xticks(x)
    ax2.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(out / "rank_profile.png", dpi=150, bbox_inches="tight")
    plt.close()
 
    # ── Encoder/decoder symmetry — one row per pair ────────────────────────────
    pairs = get_symmetric_pairs(after)
    n     = len(pairs)
    fig   = plt.figure(figsize=(15, 4.5 * n))
    gs    = gridspec.GridSpec(n, 3, figure=fig, hspace=0.55, wspace=0.38)
    fig.suptitle(
        "Encoder–Decoder Spectral Symmetry (ratio → 1 supports adjoint hypothesis)",
        fontsize=13
    )
 
    for row, (ek, dk) in enumerate(pairs):
        pair_num = row   # 0 = shallowest (closest to image), n-1 = deepest
 
        S_enc_pre  = before[ek]["S"];  S_enc_post = after[ek]["S"]
        S_dec_pre  = before[dk]["S"];  S_dec_post = after[dk]["S"]
        k = min(len(S_enc_pre), len(S_dec_pre), 40)
 
        score_pre  = spectral_similarity_score(S_enc_pre,  S_dec_pre)
        score_post = spectral_similarity_score(S_enc_post, S_dec_post)
 
        label_pre  = f"init   (sim={score_pre:.3f})"
        label_post = f"trained (sim={score_post:.3f})"
 
        # Col 0: normalised spectra at init
        ax0 = fig.add_subplot(gs[row, 0])
        ax0.plot(S_enc_pre[:k] / S_enc_pre[0],  color="steelblue", label="encoder")
        ax0.plot(S_dec_pre[:k] / S_dec_pre[0],  color="tomato",    label="decoder",
                 linestyle="--")
        ax0.set_title(f"Pair {pair_num}  [{ek}  ↔  {dk}]\nINIT  {label_pre}",
                      fontsize=8)
        ax0.set_yscale("log"); ax0.grid(alpha=0.3)
        if row == 0: ax0.legend(fontsize=8)
 
        # Col 1: normalised spectra after training
        ax1 = fig.add_subplot(gs[row, 1])
        ax1.plot(S_enc_post[:k] / S_enc_post[0], color="steelblue", label="encoder")
        ax1.plot(S_dec_post[:k] / S_dec_post[0], color="tomato",    label="decoder",
                 linestyle="--")
        ax1.set_title(f"Pair {pair_num}  [{ek}  ↔  {dk}]\nTRAINED  {label_post}",
                      fontsize=8)
        ax1.set_yscale("log"); ax1.grid(alpha=0.3)
 
        # Col 2: ratio before (gray) and after (purple) with score annotation
        ax2 = fig.add_subplot(gs[row, 2])
        ratio_pre  = S_enc_pre[:k]  / (S_dec_pre[:k]  + 1e-10)
        ratio_post = S_enc_post[:k] / (S_dec_post[:k] + 1e-10)
        ax2.plot(ratio_pre,  color="gray",   lw=1.5, alpha=0.7, label=f"init ({score_pre:.3f})")
        ax2.plot(ratio_post, color="purple", lw=1.5, linestyle="--",
                 label=f"trained ({score_post:.3f})")
        ax2.axhline(1.0, color="black", lw=0.8, linestyle=":")
        ax2.set_title(f"Pair {pair_num}  σ(enc)/σ(dec) — closer to 1 = more adjoint-like",
                      fontsize=8)
        ax2.grid(alpha=0.3); ax2.legend(fontsize=8)
 
    plt.savefig(out / "encoder_decoder_symmetry.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plots saved to {out}/")

In [ ]:
def run_training(
    path:          str,
    num_epochs:    int   = 100,
    lr:            float = 1e-3,
    batch_size:    int   = 4,
    depth:         int   = 3,
    base_channels: int   = 64,
    print_interval:int   = 10,
    out_dir:       str   = ".",
    device:        str   = None,
    model:         UNet  = None,
):
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    print(f"Device: {device}")
 
    # ── Datasets ──────────────────────────────────────────────────────────────
    train_transform = get_transforms(img_size=512, mode="train")
    val_transform   = get_transforms(img_size=512, mode="val")
 
    full_path   = path + "/unmodified-data/"
    full_train  = ISBIDataset(full_path, split="train", transform=train_transform)
    test_ds     = ISBIDataset(full_path, split="test",  transform=val_transform)
 
    # Keep a clean val set (no augmentation) — use the base dataset with val transform
    train_size  = int(0.8 * len(full_train))
    val_size    = len(full_train) - train_size
    train_ds, val_ds_raw = random_split(
        full_train, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
 
    # Apply val transforms to validation split
    val_ds_clean = ISBIDataset(full_path, split="train", transform=val_transform)
    val_indices  = val_ds_raw.indices
    val_ds       = torch.utils.data.Subset(val_ds_clean, val_indices)
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2)
 
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
 
    # ── Model ─────────────────────────────────────────────────────────────────
    if model is None:
        model = UNet(
            in_channels=1, out_channels=2,
            base_channels=base_channels, depth=depth, bilinear=True
        )
    model = model.to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
 
    # ── Spectral snapshot: BEFORE ──────────────────────────────────────────────
    model.save_weights_for_analysis(f"{out_dir}/initial_weights.pt")
    spectra_before = snapshot_spectra(model)
 
    # ── Optimizer / loss ───────────────────────────────────────────────────────
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion = CombinedLoss(alpha=0.5)   # 50% CE + 50% Dice
 
    # ── Training loop ──────────────────────────────────────────────────────────
    losses = []
    for epoch in range(1, num_epochs + 1):
        # Train
        model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()
 
        # Validate
        model.eval()
        val_loss_total = 0.0
        val_dice_total = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model(imgs)
                val_loss_total += criterion(logits, masks).item()
                val_dice_total += dice_score(logits, masks)
        vl = val_loss_total / len(val_loader)
        vd = val_dice_total / len(val_loader)
        losses.append({"train_loss": train_loss, "val_loss": vl, "val_dice": vd})
 
        if epoch % print_interval == 0:
            print(f"Epoch {epoch:3d}/{num_epochs}  "
                  f"train={train_loss:.4f}  val={vl:.4f}  dice={vd:.4f}")
 
    # ── Spectral snapshot: AFTER ───────────────────────────────────────────────
    model.save_weights_for_analysis(f"{out_dir}/trained_weights.pt")
    spectra_after = snapshot_spectra(model)
 
    # Print similarity scores per pair
    pairs = get_symmetric_pairs(spectra_after)
    print("\nSymmetry scores (1.0 = perfectly adjoint-like):")
    print(f"  {'Pair':>5}  {'Encoder key':<22} {'Decoder key':<22} "
          f"{'sim (init)':>10} {'sim (trained)':>13} {'Δ':>6}")
    print("  " + "─" * 80)
    for i, (ek, dk) in enumerate(pairs):
        s_pre  = spectral_similarity_score(spectra_before[ek]["S"], spectra_before[dk]["S"])
        s_post = spectral_similarity_score(spectra_after[ek]["S"],  spectra_after[dk]["S"])
        delta  = s_post - s_pre
        sign   = "+" if delta >= 0 else ""
        print(f"  {i:>5}  {ek:<22} {dk:<22} {s_pre:>10.4f} {s_post:>13.4f} {sign}{delta:.4f}")
 
    plot_all(spectra_before, spectra_after, losses, out_dir=out_dir)
    return model, spectra_before, spectra_after, losses

In [ ]:
model, spectra_before, spectra_after, losses = run_training(
    path=path,            
    num_epochs=200,
    depth=3,
    base_channels=64,
    out_dir="./results",
)

Device: cuda
Train: 24 | Val: 6 | Test: 30
Params: 4,281,154
Saved weights → ./results/initial_weights.pt
Epoch  10/200  train=0.2090  val=0.1885  dice=0.9246
Epoch  20/200  train=0.1886  val=0.1750  dice=0.9315
Epoch  30/200  train=0.1636  val=0.1487  dice=0.9402
Epoch  40/200  train=0.1641  val=0.1466  dice=0.9420
Epoch  50/200  train=0.1560  val=0.1530  dice=0.9386
Epoch  60/200  train=0.1474  val=0.1405  dice=0.9440
Epoch  70/200  train=0.1478  val=0.1367  dice=0.9456
Epoch  80/200  train=0.1398  val=0.1352  dice=0.9456
Epoch  90/200  train=0.1386  val=0.1457  dice=0.9430
Epoch 100/200  train=0.1362  val=0.1346  dice=0.9472
Epoch 110/200  train=0.1295  val=0.1397  dice=0.9442
Epoch 120/200  train=0.1276  val=0.1364  dice=0.9479
Epoch 130/200  train=0.1238  val=0.1318  dice=0.9480
Epoch 140/200  train=0.1202  val=0.1359  dice=0.9466
Epoch 150/200  train=0.1171  val=0.1370  dice=0.9457
Epoch 160/200  train=0.1147  val=0.1392  dice=0.9461
Epoch 170/200  train=0.1111  val=0.1375  dice=

In [20]:
def match_singular_values(W_source: np.ndarray, W_target: np.ndarray) -> np.ndarray:
    """
    Return a new matrix with:
      - The LEFT and RIGHT singular vectors of W_target  (directions preserved)
      - The singular values of W_source                  (spectrum transplanted)
 
    Formally: if W_target = U S V^T, return U S_source V^T
    where S_source is interpolated/truncated to match the rank of W_target.
 
    Args
    ----
    W_source : 2D array whose singular value SPECTRUM we want to copy
    W_target : 2D array whose singular DIRECTIONS we want to preserve
 
    Returns
    -------
    W_new : same shape as W_target, with W_source's spectrum
    """
    U_t, S_t, Vh_t = np.linalg.svd(W_target, full_matrices=False)
    _,   S_s, _    = np.linalg.svd(W_source, full_matrices=False)
 
    k_t = len(S_t)
    k_s = len(S_s)
 
    # Interpolate S_source to the rank of W_target if sizes differ
    if k_s != k_t:
        indices = np.linspace(0, k_s - 1, k_t)
        S_s = np.interp(indices, np.arange(k_s), S_s)
 
    # Rescale: keep the overall Frobenius norm of W_target so learning rates
    # don't need adjustment, but reshape the spectral decay to match W_source
    norm_ratio = np.linalg.norm(S_t) / (np.linalg.norm(S_s) + 1e-10)
    S_new      = S_s * norm_ratio          # same Frobenius norm, new spectral shape
 
    W_new = (U_t * S_new) @ Vh_t          # reconstruct: U diag(S_new) V^T
    return W_new
 
 
def get_conv_weight_2d(conv: nn.Conv2d) -> np.ndarray:
    """Extract and flatten conv weight: (out, in, kH, kW) → (out, in*kH*kW)."""
    w = conv.weight.detach().cpu().numpy()
    return w.reshape(w.shape[0], -1)
 
 
def set_conv_weight_2d(conv: nn.Conv2d, W_new: np.ndarray):
    """Reshape flat 2D matrix back into conv weight and assign."""
    original_shape = conv.weight.shape
    W_reshaped     = W_new.reshape(original_shape)
    with torch.no_grad():
        conv.weight.copy_(torch.from_numpy(W_reshaped.astype(np.float32)))

 
def apply_spectral_init(model: UNet, depth: int, verbose: bool = True):
    """
    For each symmetric encoder/decoder pair, rescale the decoder's conv
    singular values to match the encoder's spectrum.
 
    Pair mapping (shallowest to deepest):
        encoder_L{i}  ↔  decoder_L{depth-1-i}
 
    Modifies model weights IN PLACE.
 
    Args
    ----
    model   : UNet instance (before training)
    depth   : must match the depth used to construct the model
    verbose : print similarity scores before and after
    """ 
    print("Applying spectral initialization...\n")
 
    for i in range(depth):
        enc_block = model.encoder[i]
        dec_block = model.decoder[depth - 1 - i]
 
        # Each block has a DoubleConv with conv1 and conv2
        enc_conv = enc_block if isinstance(enc_block, nn.Sequential) else enc_block
        # Navigate to the actual DoubleConv
        enc_dc   = enc_block if hasattr(enc_block, 'block') else enc_block.conv
        dec_dc   = dec_block.conv
 
        for conv_name in ["conv1", "conv2"]:
            enc_conv_layer = getattr(enc_dc.block, conv_name)
            dec_conv_layer = getattr(dec_dc.block, conv_name)
 
            W_enc = get_conv_weight_2d(enc_conv_layer)
            W_dec = get_conv_weight_2d(dec_conv_layer)
 
            sim_before = spectral_similarity_score(
                np.linalg.svd(W_enc, full_matrices=False)[1],
                np.linalg.svd(W_dec, full_matrices=False)[1]
            )
 
            W_dec_new  = match_singular_values(W_source=W_enc, W_target=W_dec)
            set_conv_weight_2d(dec_conv_layer, W_dec_new)
 
            sim_after = spectral_similarity_score(
                np.linalg.svd(W_enc, full_matrices=False)[1],
                np.linalg.svd(W_dec_new, full_matrices=False)[1]
            )
 
            if verbose:
                pair_label = f"enc_L{i} / dec_L{depth-1-i} / {conv_name}"
                print(f"  {pair_label:<35}  "
                      f"sim before={sim_before:.4f}  →  after={sim_after:.4f}")
 
    print("\nSpectral init complete. Train as normal.")
 
 
def compare_spectra_table(model_standard: UNet, model_spectral: UNet, depth: int):
    """
    Print a side-by-side comparison of per-pair similarity scores for a
    standard randomly initialized model vs a spectrally initialized one.
    Useful as a sanity check before training.
    """
    print(f"\n{'Pair':<6} {'Layer':<10} {'Standard init':>15} {'Spectral init':>15}")
    print("─" * 50)
 
    for i in range(depth):
        for conv_name in ["conv1", "conv2"]:
            enc_dc_std  = (model_standard.encoder[i]
                           if hasattr(model_standard.encoder[i], 'block')
                           else model_standard.encoder[i].conv)
            dec_dc_std  = model_standard.decoder[depth - 1 - i].conv
 
            enc_dc_spc  = (model_spectral.encoder[i]
                           if hasattr(model_spectral.encoder[i], 'block')
                           else model_spectral.encoder[i].conv)
            dec_dc_spc  = model_spectral.decoder[depth - 1 - i].conv
 
            def sim(enc_dc, dec_dc):
                We = get_conv_weight_2d(getattr(enc_dc.block, conv_name))
                Wd = get_conv_weight_2d(getattr(dec_dc.block, conv_name))
                Se = np.linalg.svd(We, full_matrices=False)[1]
                Sd = np.linalg.svd(Wd, full_matrices=False)[1]
                return spectral_similarity_score(Se, Sd)
 
            s_std = sim(enc_dc_std, dec_dc_std)
            s_spc = sim(enc_dc_spc, dec_dc_spc)
 
            print(f"  {i:<4} {conv_name:<10} {s_std:>15.4f} {s_spc:>15.4f}")

In [22]:
!mkdir ./results/random/
!mkdir ./results/spectral/

In [25]:
depth = 3

# Standard init — train this as your baseline
model_standard = UNet(in_channels=1, out_channels=2,
                      base_channels=64, depth=depth, bilinear=True).to(device)

# Spectral init — copy architecture, then transplant spectra
model_spectral = copy.deepcopy(model_standard)   # same random directions
apply_spectral_init(model_spectral, depth=depth, verbose=True)

# Sanity check before training
compare_spectra_table(model_standard, model_spectral, depth=depth)

# Then train both with identical hyperparameters and compare loss curves + Dice
model_standard, spectra_before, spectra_after, losses = run_training(
    path=path,            
    num_epochs=100,
    depth=depth,
    base_channels=64,
    out_dir="./results/random/",
    model=model_standard
)

model_spectral, spectra_before, spectra_after, losses = run_training(
    path=path,            
    num_epochs=100,
    depth=depth,
    base_channels=64,
    out_dir="./results/spectral/",
    model=model_spectral
)

Applying spectral initialization...

  enc_L0 / dec_L2 / conv1              sim before=0.0000  →  after=0.1093
  enc_L0 / dec_L2 / conv2              sim before=0.9914  →  after=0.9987
  enc_L1 / dec_L1 / conv1              sim before=0.8925  →  after=0.9975
  enc_L1 / dec_L1 / conv2              sim before=0.8603  →  after=0.8396
  enc_L2 / dec_L0 / conv1              sim before=0.8922  →  after=0.9993
  enc_L2 / dec_L0 / conv2              sim before=0.8586  →  after=0.8372

Spectral init complete. Train as normal.

Pair   Layer        Standard init   Spectral init
──────────────────────────────────────────────────
  0    conv1               0.0000          0.1093
  0    conv2               0.9914          0.9987
  1    conv1               0.8925          0.9975
  1    conv2               0.8603          0.8396
  2    conv1               0.8922          0.9993
  2    conv2               0.8586          0.8372
Device: cuda
Train: 24 | Val: 6 | Test: 30
Params: 4,281,154
Saved weights 